# Бейзлайны детекции бабочек

Сравниваем Grounding DINO Tiny, OWLv2 Base и Qwen3‑VL‑2B‑Instruct на одних и тех же 400 фото. По умолчанию используем сохранённые предсказания и сверяем их с сырыми выходами. Для нового инференса установи зависимости для CPU или CUDA по README и включи `RERUN_ALL=True`. Каждая модель запускается своей ячейкой.

In [1]:
import sys
from pathlib import Path
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = (cwd, *cwd.parents, Path("/kaggle/working/data-labeling-shad"))
ROOT = next((p for p in candidates if (p / "model_comparison/workflow.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Открой ноутбук из клона репозитория или склонируй его в /kaggle/working/data-labeling-shad")
sys.path.insert(0, str(ROOT))
print("Корень репозитория найден")

Корень репозитория найден


In [2]:
from model_comparison.workflow import baseline, group_table, leaderboard, load_validation, measure, check_production_runtime

photos, golden = load_validation()
print(f"Фото: {len(photos)} | С бабочками: {sum(bool(r['boxes']) for r in golden.values())}"
      f" | Рамок в эталоне: {sum(len(r['boxes']) for r in golden.values())}")

Фото: 400 | С бабочками: 197 | Рамок в эталоне: 218


In [3]:
RERUN_ALL = False
DEVICE = check_production_runtime("auto") if RERUN_ALL else None
RUN_DIR = ROOT / "runs/production_validation" / (DEVICE or "replay")
GROUP_COLUMNS = ["n_photos", "f1_50", "f1_75", "precision_50", "recall_50",
                "fp_50", "fn_50", "negative_photos_with_fp", "fp_on_negative_photos",
                "technical_errors"]
TEAM_COLUMNS = GROUP_COLUMNS + ["inference_seconds", "ms_per_photo"]
print("Источник:", f"новый инференс на {DEVICE}" if RERUN_ALL else "сохранённые запуски")

Источник: сохранённые запуски


Метрики считаются по объектам. Для каждого фото при заданном IoU строится максимальное паросочетание между эталонными и предсказанными рамками. TP, FP и FN суммируются перед расчётом метрик. В группах без эталонных объектов основными показателями служат количество кадров с FP и суммарное число FP. Пустые значения времени означают, что у опубликованного прогона нет сырого лога с таймингом. Показанное время инференса не включает загрузку фотографий и весов.

## Grounding DINO Tiny

In [4]:
dino_pred, dino_seconds = baseline(
    "grounding_dino", photos, rerun=RERUN_ALL, device=DEVICE, output_dir=RUN_DIR / "grounding_dino",
)
print(f"Grounding DINO Tiny: {len(dino_pred)} предсказаний")

Grounding DINO Tiny: 400 предсказаний


In [5]:
dino_result = measure("grounding_dino", photos, golden, dino_pred, dino_seconds)
display(leaderboard(dino_result)[TEAM_COLUMNS].round(4))
display(group_table(dino_result)[GROUP_COLUMNS].round(4))

,n_photos,f1_50,f1_75,precision_50,recall_50,fp_50,fn_50,negative_photos_with_fp,fp_on_negative_photos,technical_errors,inference_seconds,ms_per_photo
run,,,,,,,,,,,,
grounding_dino,400,0.3788,0.351,0.2375,0.9358,655,14,182,341,0,199.2971,498.2427


n_photos  ...  technical_errors
run            group                         ...                  
grounding_dino butterfly                191  ...                 0
               insect                   101  ...                 0
               multiple_butterfly         6  ...                 0
               no_butterfly             102  ...                 0

[4 rows x 10 columns]

## OWLv2 Base

In [6]:
owl_pred, owl_seconds = baseline(
    "owlv2", photos, rerun=RERUN_ALL, device=DEVICE, output_dir=RUN_DIR / "owlv2",
)
print(f"OWLv2 Base: {len(owl_pred)} предсказаний")

OWLv2 Base: 400 предсказаний


In [7]:
owl_result = measure("owlv2", photos, golden, owl_pred, owl_seconds)
display(leaderboard(owl_result)[TEAM_COLUMNS].round(4))
display(group_table(owl_result)[GROUP_COLUMNS].round(4))

,n_photos,f1_50,f1_75,precision_50,recall_50,fp_50,fn_50,negative_photos_with_fp,fp_on_negative_photos,technical_errors,inference_seconds,ms_per_photo
run,,,,,,,,,,,,
owlv2,400,0.476,0.3486,0.3146,0.9771,464,5,103,191,0,337.002,842.505


n_photos  ...  technical_errors
run   group                         ...                  
owlv2 butterfly                191  ...                 0
      insect                   101  ...                 0
      multiple_butterfly         6  ...                 0
      no_butterfly             102  ...                 0

[4 rows x 10 columns]

## Qwen3-VL-2B-Instruct

In [8]:
qwen_pred, qwen_seconds = baseline(
    "qwen", photos, rerun=RERUN_ALL, device=DEVICE, output_dir=RUN_DIR / "qwen",
)
print(f"Qwen3-VL-2B-Instruct: {len(qwen_pred)} предсказаний")

Qwen3-VL-2B-Instruct: 400 предсказаний


In [9]:
qwen_result = measure("qwen", photos, golden, qwen_pred, qwen_seconds)
display(leaderboard(qwen_result)[TEAM_COLUMNS].round(4))
display(group_table(qwen_result)[GROUP_COLUMNS].round(4))

,n_photos,f1_50,f1_75,precision_50,recall_50,fp_50,fn_50,negative_photos_with_fp,fp_on_negative_photos,technical_errors,inference_seconds,ms_per_photo
run,,,,,,,,,,,,
qwen,400,0.6508,0.5841,0.4976,0.9404,207,13,202,206,0,953.0246,2382.5615


n_photos  ...  technical_errors
run  group                         ...                  
qwen butterfly                191  ...                 0
     insect                   101  ...                 0
     multiple_butterfly         6  ...                 0
     no_butterfly             102  ...                 0

[4 rows x 10 columns]

## Сравнение

In [10]:
baseline_table = leaderboard(dino_result, owl_result, qwen_result)
display(baseline_table[TEAM_COLUMNS].round(4))

,n_photos,f1_50,f1_75,precision_50,recall_50,fp_50,fn_50,negative_photos_with_fp,fp_on_negative_photos,technical_errors,inference_seconds,ms_per_photo
run,,,,,,,,,,,,
qwen,400,0.6508,0.5841,0.4976,0.9404,207,13,202,206,0,953.0246,2382.5615
owlv2,400,0.4760,0.3486,0.3146,0.9771,464,5,103,191,0,337.0020,842.5050
grounding_dino,400,0.3788,0.3510,0.2375,0.9358,655,14,182,341,0,199.2971,498.2427


In [11]:
table_dir = RUN_DIR / "tables" if RERUN_ALL else ROOT / "runs/tables"
table_dir.mkdir(parents=True, exist_ok=True)
baseline_table.to_csv(table_dir / "leaderboard_baseline.csv")
print("Сохранено:", (table_dir / "leaderboard_baseline.csv").relative_to(ROOT))

Сохранено: runs/tables/leaderboard_baseline.csv


На опубликованных запусках исходный Qwen лидирует по F1@0.50 (0.6508). На фото с одной бабочкой он сопоставляет все 191 эталонную рамку при IoU ≥ 0.50, но на фото с несколькими бабочками пропускает 13 из 27 объектов. OWLv2 даёт наибольший общий recall (0.9771), хотя добавляет 464 FP. Grounding DINO выдаёт 655 FP, из которых 300 приходятся на группу с одиночной бабочкой. Это мотивирует проверку удаления дублей. У исходного Qwen лишняя рамка появляется на 202 из 203 фото без бабочек, поэтому дальше проверяем более строгий промпт.